# Revision Overview: What Has Been Done and What It Shows

This notebook reviews the work on the revision of "From Graph k-Coloring to Grover's Circuits via Logic Synthesis" in
one place. It follows the argument of the revised paper:

1. **The problem.** Graph coloring as a Grover search, its encoding and the three ways of preparing the search space.
2. **Three errors in the original evaluation**, each shown on a small example: the diffusion operator, the vertex
   labels of the baseline circuits, and the counting of ancilla qubits.
3. **The explicit cost model** that replaces the manuscript's estimates: decompositions of multi-controlled gates,
   exact Clifford+T gadgets, and measurement-based uncomputation.
4. **The corrected comparison** of every synthesis method with every baseline on the paper's nine graphs.
5. **Why our oracle ties the Belletti-Simple baseline**, and where the width claims of the manuscript came from.
6. **Optimizing the synthesized oracle**: fewer AND gates, balanced logic, and Caterpillar's mapping strategies.
7. **Cheaper preparation of the valid colors**: exact repeat-until-success rotations, Hamming-weight phasing and
   amplitude amplification, priced on the complete search.
8. **Key results**, computed from the sections above, and the work that remains.

Every number is computed in this run, either from the saved nine-graph study named by `LOGIQUE_STUDY_RUN` or from
circuits built here. Figures are saved as PNG and SVG files next to the executed notebook.

## Setup

In [ ]:
from dataclasses import replace
from pathlib import Path
import json
import math
import os

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import SVG, Image, Markdown, display
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.ticker import LogLocator
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Operator, Statevector

from logique import encode_coloring, read_dimacs
from logique.benchmarks import analysis, costs
from logique.benchmarks import preparations as prepared
from logique.benchmarks.study import expand_oracle
from logique.circuits import preparation as prep
from logique.circuits.search import baseline_instance, logique_instance
from logique.paths import native_executable, new_dump_directory
from logique.synthesis import synthesize

ROOT = Path(os.environ.get("LOGIQUE_WORKSPACE", Path.cwd())).resolve()
if "LOGIQUE_STUDY_RUN" not in os.environ:
    raise RuntimeError("Set LOGIQUE_STUDY_RUN to the folder of the completed nine-graph study")
RUN = (ROOT / os.environ["LOGIQUE_STUDY_RUN"]).resolve()
if "LOGIQUE_OUTPUT" in os.environ:
    OUT = Path(os.environ["LOGIQUE_OUTPUT"]).resolve()
else:
    OUT = new_dump_directory("revision-overview", root=ROOT / "dump")
FIGURES = OUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
EXECUTABLE = native_executable(
    os.environ.get("LOGIQUE_NATIVE_EXECUTABLE")
    or ROOT / "external/native/boolean_synthesis/build/boolean_synthesis"
)
GRAPH_DIR = ROOT / "datasets/graphs/benchmarks"

rows, manifest = analysis.load_study(RUN)
settings = manifest["settings"]
GRAPHS = [Path(p).stem for p in settings["inputs"]]
LOGIQUE = [m for m in settings["methods"] if not m.startswith("sb_")]
BASELINES = [m for m in settings["methods"] if m.startswith("sb_")]
POLICIES = settings["mcx_policies"]
REGIMES = settings["regimes"]
problems = {g: replace(read_dimacs(GRAPH_DIR / f"{g}.col"), precolored={}) for g in GRAPHS}

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, SECONDARY, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "font.size": 9,
        "axes.edgecolor": AXIS,
        "axes.labelcolor": SECONDARY,
        "xtick.color": SECONDARY,
        "ytick.color": SECONDARY,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.6,
        "axes.axisbelow": True,
    }
)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
KEY = {}  # numbers quoted in the summary of Section 8


def show_figure(figure, stem: str) -> None:
    """Save a Matplotlib figure as PNG and SVG, then display it."""
    figure.savefig(FIGURES / f"{stem}.svg", bbox_inches="tight")
    figure.savefig(FIGURES / f"{stem}.png", dpi=140, bbox_inches="tight")
    display(Image(filename=str(FIGURES / f"{stem}.png")))
    plt.close(figure)


def show_svg(svg: bytes, stem: str) -> None:
    (FIGURES / f"{stem}.svg").write_bytes(svg)
    display(SVG(data=svg))


def log_axis(axis):
    axis.set_major_locator(LogLocator(subs=(1.0, 2.0, 5.0)))
    axis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
    axis.set_minor_formatter(plt.NullFormatter())


def role_groups(instance, circuit):
    """Wire groups for drawing: one register per vertex, then each role."""
    groups = [
        (f"v{vertex}", [instance.data[i] for i in inputs])
        for vertex, inputs in instance.encoding.input_mapping.items()
    ]
    role = "ancillas" if instance.method.startswith("sb_") else "workspace"
    groups.append(("e" if role == "ancillas" else "w", instance.roles[role]))
    groups.append(("a", list(range(instance.oracle.num_qubits, circuit.num_qubits))))
    groups.append(("out", instance.roles["output"]))
    return [(name, wires) for name, wires in groups if wires]


def labelled(circuit, groups):
    """Copy a circuit onto named registers so that drawings show qubit roles."""
    registers = [QuantumRegister(len(wires), name) for name, wires in groups]
    drawn = QuantumCircuit(*registers, global_phase=circuit.global_phase)
    position = {}
    for register, (_, wires) in zip(registers, groups):
        position.update(zip(wires, register))
    for inst in circuit.data:
        drawn.append(inst.operation, [position[circuit.find_bit(q).index] for q in inst.qubits])
    return drawn


def expanded(instance, policy="throwaway_tree"):
    """Expand an oracle as the study does: MCX policy, then relative-phase pairs."""
    circuit, _ = expand_oracle(instance, policy)
    return costs.paired_relative_phase_toffolis(circuit)


ft = rows[rows.metric_level == "fault-tolerant"]
search = analysis.search_costs(rows)
print(f"Study {manifest['run_id']}: {len(manifest['cases'])} cases, {len(GRAPHS)} graphs, "
      f"{len(LOGIQUE)} synthesis methods, {len(BASELINES)} baselines, policies {', '.join(POLICIES)}")
print("Artifacts:", OUT)

## 1. The problem

A graph with vertices V and edges E is k-colorable if each vertex can receive one of k colors so that adjacent
vertices differ. Each vertex holds its color in c = ⌈log₂ k⌉ qubits, least-significant bit first, so when k is not a
power of two some of the 2^c codes are unused. Grover's algorithm repeats an **oracle**, which marks proper
colorings, and a **diffusion operator**, which reflects about the starting state. Three settings are compared
throughout:

| Setting | Starting state | What the oracle checks |
|---|---|---|
| **R1** | all 2^c codes per vertex (Hadamard gates) | edges and the unused codes |
| **R2** | only the k valid codes per vertex | edges and the unused codes |
| **R3** | only the k valid codes per vertex | edges only (the unused codes never occur) |

Starting only in valid codes shrinks the search space from 2^(c·|V|) to k^|V| states, so fewer iterations are needed;
the price is a rotation per vertex in every preparation and inverse preparation. The table gives the nine graphs of
the paper, their search spaces and the optimal number of iterations.

In [ ]:
from logique.visualization.graphs import draw_colored_graph

problem = problems["K3"]
graph = problem.to_networkx()
draw_colored_graph(graph, name=str(FIGURES / "k3.png"), pos=nx.circular_layout(graph), figsize=(3.2, 2.8), node_size=1200)
plt.gcf().axes[0].set_title("K3 with three colors")
show_figure(plt.gcf(), "k3")

domains = (
    rows[rows.component == "search"]
    .drop_duplicates(["benchmark", "regime"])
    .pivot(index="benchmark", columns="regime", values=["domain", "iterations"])
)
overview = pd.DataFrame(
    {
        "vertices": {g: len(p.vertices) for g, p in problems.items()},
        "edges": {g: len(p.edges) for g, p in problems.items()},
        "colors k": {g: p.colors for g, p in problems.items()},
        "bits per vertex": {g: max(1, (p.colors - 1).bit_length()) for g, p in problems.items()},
        "proper colorings": rows.drop_duplicates("benchmark").set_index("benchmark").solutions.astype("Int64"),
    }
).reindex(GRAPHS)
for regime in ("R1", "R2"):
    overview[f"search space {regime}"] = domains[("domain", regime)].map(lambda v: f"{float(v):.2e}" if v == v else "")
    overview[f"iterations {regime}"] = domains[("iterations", regime)].astype("Int64")
display(overview)
display(Markdown("myciel5 has too many colorings to count, so its searches are compared per iteration."))

The oracle is synthesized from a Boolean description of the coloring constraints. Logique writes the predicate as an
Xor-And-Inverter Graph (XAG): every AND node becomes a Toffoli-type gate, while XOR and NOT nodes are free (CNOT and X
gates). The two XAGs below are for K₃: the edge-only predicate (R3) and the complete predicate with the checks of the
unused code (R1 and R2).

In [ ]:
from logique.visualization.boolean import graphviz_graph

labels = [None] * encode_coloring(problem).input_count
for vertex, inputs in encode_coloring(problem).input_mapping.items():
    for bit, i in enumerate(inputs):
        labels[i] = f"x_{vertex},{bit}"
for title, encoding, stem in (
    ("Edge-only predicate (R3)", encode_coloring(replace(problem, colors=4)), "xag-edges"),
    ("Complete predicate (R1, R2)", encode_coloring(problem), "xag-complete"),
):
    network = synthesize(encoding, "xag", executable=EXECUTABLE).metadata["network"]
    show_svg(graphviz_graph(network, title, input_labels=labels).pipe(format="svg"), stem)

## 2. Three errors in the original evaluation

**The diffusion operator.** The manuscript prepares only the valid colors but its benchmark generator reflected about
the Hadamard state, which covers every code. On K₃ with three colors this leaks probability into the unused code and
roughly halves the success probability after one iteration. Reflecting about the actual preparation fixes it.

In [ ]:
from logique.circuits.grover import kickback_grover_circuit

edge_only = encode_coloring(replace(problem, colors=4))
oracle = synthesize(edge_only, "xag", executable=EXECUTABLE)
inputs = oracle.metadata["input_qubits"]
vertex_preparation = prep.shukla_vedula(3)
preparation = QuantumCircuit(len(inputs))
for start in range(0, len(inputs), 2):
    preparation.compose(vertex_preparation, [start, start + 1], inplace=True)
hadamard = QuantumCircuit(len(inputs))
hadamard.h(range(len(inputs)))
matching = kickback_grover_circuit(oracle, preparation=preparation, iterations=1)
historical = kickback_grover_circuit(oracle, preparation=preparation, reflection_preparation=hadamard, iterations=1)

assignments = [edge_only.decode(b) for b in range(1 << len(inputs))]
unused = np.array([any(c >= 3 for c in a.values()) for a in assignments])
proper = np.array([problem.is_valid(a) for a in assignments])
groups = {"proper coloring": proper, "valid codes, conflict": ~unused & ~proper, "uses the unused code": unused}
reflections = pd.DataFrame(
    {
        "before the iteration": {k: float(Statevector(preparation).probabilities()[m].sum()) for k, m in groups.items()},
        "reflection about the preparation": {k: float(Statevector(matching).probabilities(inputs)[m].sum()) for k, m in groups.items()},
        "historical Hadamard reflection": {k: float(Statevector(historical).probabilities(inputs)[m].sum()) for k, m in groups.items()},
    }
)
display(reflections.round(3))
figure, ax = plt.subplots(figsize=(7.5, 3))
reflections.plot.bar(ax=ax, rot=0, color=[SERIES[3], SERIES[0], SERIES[1]], width=0.75, edgecolor="white")
ax.set_ylabel("Probability")
ax.set_xlabel("")
ax.grid(axis="x", visible=False)
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))
show_figure(figure, "reflection")
KEY["k3 success"] = reflections.loc["proper coloring"].iloc[1:].to_dict()

**Vertex labels of the baselines.** The baseline package uses each vertex label as a register index. The historical
replay passed labels starting at 1, so the Saha et al. variant skipped every edge of the last vertex. On a
three-vertex path with two colors, one iteration should find one of the two proper colorings with certainty; with the
historical labels it does no better than guessing.

In [ ]:
from contextlib import redirect_stdout
import io

from logique.synthesis.saha_belletti import VARIANTS, comparison_circuit

path = read_dimacs(ROOT / "datasets/graphs/execution/linear3.col")
alternating = (0b010, 0b101)
with redirect_stdout(io.StringIO()):  # the baseline package prints its iteration count
    labels_table = pd.DataFrame(
    {
        name: {
            variant: float(
                sum(
                    Statevector(comparison_circuit(path, variant, iterations=1, preserve_labels=keep)).probabilities([0, 1, 2])[b]
                    for b in alternating
                )
            )
            for variant in VARIANTS
        }
        for name, keep in (("historical labels", True), ("normalized labels", False))
    }
    )
display(labels_table.round(3))

**Counting ancilla qubits.** The manuscript counted the baselines' ancillas with a rule of its own: n − 2 ancillas
for every n-control gate except the last, summed over the whole oracle, whereas Logique's circuits were counted by the
qubits they allocate. The rule is wrong in both directions. The baseline undoes each gate in mirror order, which frees
its ancillas for reuse, so summing over all gates counts them repeatedly; and the last gate's ancillas were left out.
The correct width is the *peak* number of qubits in use, which depends on how the multi-controlled gates are
decomposed. The table applies the historical rule and the peak count to the Belletti-Simple oracle with its
unused-code checks (the setting of the manuscript's baselines), next to Logique's oracle. Under the historical rule
the baseline looks wider than Logique's oracle on every graph with c ≥ 3; counted at its peak with its best
decomposition, it is narrower on every graph.

In [ ]:
oracle_rows = ft[ft.component == "oracle"]
width_rows = []
for g in GRAPHS:
    instance = baseline_instance(problems[g], "simple", "R2")
    gates = [inst.operation.num_ctrl_qubits for inst in instance.oracle.data if costs.is_mcx(inst.operation, 3)]
    summed = instance.oracle.num_qubits + sum(n - 2 for n in gates[:-1])
    simple = oracle_rows[(oracle_rows.benchmark == g) & (oracle_rows.method == "sb_simple") & (oracle_rows.regime == "R2")]
    xag = oracle_rows[(oracle_rows.benchmark == g) & (oracle_rows.method == "xag") & (oracle_rows.regime == "R3")]
    width_rows.append(
        {
            "graph": g,
            "Belletti-Simple, historical rule": summed,
            "Belletti-Simple, peak (throwaway tree)": int(simple[simple.mcx_policy == "throwaway_tree"].qubits.iloc[0]),
            "Belletti-Simple, peak (best decomposition)": int(simple.qubits.min()),
            "Logique XAG (edge-only)": int(xag.qubits.iloc[0]),
        }
    )
widths = pd.DataFrame(width_rows).set_index("graph")
display(widths)
figure, ax = plt.subplots(figsize=(9, 3.4))
x = np.arange(len(widths))
for i, (column, color) in enumerate(zip(widths.columns, [SERIES[1], SERIES[3], SERIES[2], SERIES[0]])):
    ax.bar(x + (i - 1.5) * 0.2, widths[column], width=0.2, color=color, edgecolor="white", label=column)
ax.set_yscale("log")
log_axis(ax.yaxis)
ax.set_xticks(x, widths.index, rotation=30, ha="right")
ax.set_ylabel("Qubits")
ax.grid(axis="x", visible=False)
ax.legend(frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.25))
show_figure(figure, "width-accounting")

## 3. The explicit cost model

Every multi-controlled X gate is expanded with a named construction from the literature before anything is counted,
so ancillas and T gates come from explicit circuits. The drawings show three constructions for four controls:

- the **throwaway tree** of the manuscript's Figure 1, which leaves its ancillas in a garbage state and is valid only
  when the gate is later undone by its mirror;
- the **restoring tree**, which uncomputes its intermediate ANDs with cheap relative-phase Toffoli gates;
- **Khattar and Gidney's** construction with two clean ancillas, which uses conditionally clean ancillas.

In [ ]:
gate = QuantumCircuit(5)
gate.mcx([0, 1, 2, 3], 4)
for method in ("throwaway_tree", "clean_tree", "2_clean_kg24"):
    circuit, info = costs.decompose_mcx(gate, method)
    display(Markdown(f"**{method}** ({info['mcx_reference']}): {info['appended_ancillas']} ancilla(s)"))
    show_figure(circuit.draw("mpl", fold=40), f"mcx-{method}")

The chart compares the constructions as the number of controls grows. Ancilla-free constructions pay heavily in T
gates; constructions with a few clean ancillas stay close to the restoring tree, which uses n − 2 ancillas.

In [ ]:
shown = ["throwaway_tree", "clean_tree", "2_clean_kg24", "1_clean_kg24", "n_dirty_i15", "noaux_v24"]
isolated = pd.DataFrame(
    [costs.isolated_mcx_costs(method, n) for method in shown for n in (3, 4, 6, 8, 12, 16, 20)]
)
figure, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for color, method in zip(SERIES, shown):
    part = isolated[(isolated.method == method) & (isolated.controls >= 4)]
    axes[0].plot(part.controls, part.t_count, marker="o", markersize=4, linewidth=2, color=color, label=method)
    axes[1].plot(part.controls, part.ancillas, marker="o", markersize=4, linewidth=2, color=color, label=method)
axes[0].set_yscale("log")
log_axis(axes[0].yaxis)
axes[0].set_ylabel("T gates")
axes[1].set_ylabel("Ancillas")
for ax in axes:
    ax.set_xlabel("Controls")
axes[1].legend(frameon=False, loc="upper left", fontsize=8)
show_figure(figure, "mcx-isolated")
display(Markdown("T gates by construction and number of controls"))
display(isolated.pivot(index="controls", columns="method", values="t_count")[shown])
display(Markdown("Arbitrary rotations left by the ancilla-free construction (priced separately)"))
display(isolated[isolated.method == "noaux_v24"].set_index("controls")[["rotation_count"]].T)

Counting uses fixed gadgets with exact T-counts: a Toffoli gate costs 7 T gates, and a relative-phase Toffoli gate
(or the doubly controlled Rx(±π) produced by XAG synthesis) costs 4. The cheaper gadget is allowed only where its
extra phase provably cancels; the same rules are applied to every method, so no flow gets a discount the others do
not. Uncomputing an AND gate by measurement (Gidney) costs no T gate at all; the table shows how much this saves on the
oracle, reported separately because it needs mid-circuit measurement.

In [ ]:
oracles = rows[(rows.component == "oracle") & rows.metric_level.isin(["fault-tolerant", "fault-tolerant, measurement-assisted"])]
pivot = oracles[(oracles.regime == "R3") & (oracles.mcx_policy == "clean_tree")].pivot_table(
    index=["benchmark", "method"], columns="metric_level", values="t_count"
)
pivot["saving"] = 1 - pivot["fault-tolerant, measurement-assisted"] / pivot["fault-tolerant"]
savings = pivot["saving"].unstack().reindex(index=GRAPHS)[["xag", "aig_bennett", "xag_bennett", "sb_simple", "sb_balanced"]]
display(Markdown("Share of the oracle's T gates saved by measurement-based uncomputation (R3, restoring tree)"))
display(savings.round(2))
KEY["measurement saving"] = savings.max().round(2).to_dict()

## 4. The corrected comparison

Thirteen constructions (nine synthesis methods and four baselines) were built on the nine graphs in the three
settings, every multi-controlled gate was expanded with each of five constructions, and every circuit was priced in
Clifford+T. A search is compared with the **strongest baseline** on the same graph and setting: a ratio below 1 means
fewer T gates or qubits than the best baseline. Each method uses its own best decomposition.

In [ ]:
per_iteration = analysis.iteration_costs(rows, budget=settings.get("rotation_budget", 1e-3))
comparisons = {
    metric: analysis.compare_with_baselines(per_iteration, metric, matched_policy=False)
    for metric in ("t_count", "qubits_total")
}
summary = pd.concat(
    {metric: analysis.summarize_outcomes(table) for metric, table in comparisons.items()}, names=["metric"]
).reset_index()
view = summary.pivot_table(index="method", columns=["metric", "regime"], values="geometric_mean_ratio").reindex(LOGIQUE)
display(Markdown("Geometric mean over the nine graphs of the ratio to the strongest baseline, per Grover iteration"))
display(view.round(2))
outcomes = summary[summary.method == "xag"].set_index(["metric", "regime"])[["wins", "ties", "losses"]]
display(Markdown("Wins, ties and losses of the XAG oracle against the strongest baseline"))
display(outcomes)
KEY["xag ratios"] = view.loc["xag"].round(2).to_dict()

In [ ]:
diverging = LinearSegmentedColormap.from_list("ratio", [SERIES[0], "#f0efec", SERIES[7]])
table = comparisons["t_count"]
figure, axes = plt.subplots(1, len(REGIMES), figsize=(4.3 * len(REGIMES), 0.42 * len(LOGIQUE) + 1.6), sharey=True)
for ax, regime in zip(np.atleast_1d(axes), REGIMES):
    grid = table[table.regime == regime].pivot(index="method", columns="benchmark", values="ratio")
    grid = grid.reindex(index=LOGIQUE, columns=[g for g in GRAPHS if g in grid.columns])
    ax.imshow(np.log2(grid.to_numpy(dtype=float)), cmap=diverging, norm=TwoSlopeNorm(0, -4, 4), aspect="auto")
    for (i, j), ratio in np.ndenumerate(grid.to_numpy(dtype=float)):
        if np.isfinite(ratio):
            ax.text(j, i, f"{ratio:.2f}" if ratio < 10 else f"{ratio:.0f}", ha="center", va="center", fontsize=7, color=INK)
    ax.set_xticks(range(grid.shape[1]), grid.columns, rotation=45, ha="right")
    ax.set_yticks(range(grid.shape[0]), grid.index)
    ax.set_title(regime, color=INK)
    ax.grid(False)
figure.suptitle("T gates per iteration relative to the strongest baseline (blue: fewer, red: more)", color=INK, y=1.02)
show_figure(figure, "t-ratio-heatmap")

The next chart isolates the XAG oracle, Logique's main flow: its T-count never exceeds the strongest baseline, but it
needs more qubits on every graph.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=False)
for ax, metric, title in zip(axes, ("t_count", "qubits_total"), ("T gates", "Qubits")):
    part = comparisons[metric][comparisons[metric].method == "xag"]
    for r, regime in enumerate(REGIMES):
        values = part[part.regime == regime].set_index("benchmark").ratio.reindex(GRAPHS)
        ax.scatter(np.arange(len(GRAPHS)) + (r - 1) * 0.18, values, s=40, color=SERIES[r], edgecolor="white", linewidth=0.8, label=regime, zorder=2)
    ax.axhline(1, color=SECONDARY, linewidth=1, zorder=1)
    ax.set_xticks(range(len(GRAPHS)), GRAPHS, rotation=45, ha="right")
    ax.set_title(f"{title}: XAG / strongest baseline", color=INK)
    ax.grid(axis="x", visible=False)
axes[0].legend(frameon=False, loc="lower left")
show_figure(figure, "xag-vs-baseline")

Where do the qubits go? The bars split each circuit's width into the data qubits, the output qubit, the oracle's own
workspace and the ancillas added by the decomposition (edge-only setting, Khattar–Gidney decomposition).

In [ ]:
widths_by_role = analysis.width_breakdown(rows)
policy = "2_clean_kg24" if "2_clean_kg24" in POLICIES else POLICIES[0]
shown_graphs = ["K5", "K10", "myciel4", "K20"]
methods = ["xag", "xag_lowt", "aig_bennett", *BASELINES]
roles = {"data_qubits": "data", "output_qubits": "output", "oracle_workspace": "oracle workspace", "mcx_ancillas": "decomposition ancillas"}
part = widths_by_role[(widths_by_role.regime == "R3") & (widths_by_role.mcx_policy == policy)]
figure, axes = plt.subplots(1, len(shown_graphs), figsize=(12, 3.2), sharey=True)
for ax, g in zip(axes, shown_graphs):
    data = part[part.benchmark == g].set_index("method").reindex(methods).fillna(0)
    left = np.zeros(len(data))
    for color, (role, label) in zip(SERIES, roles.items()):
        ax.barh(data.index, data[role], left=left, color=color, edgecolor="white", height=0.7, label=label)
        left += data[role].to_numpy()
    ax.invert_yaxis()
    ax.set_title(g, color=INK)
    ax.grid(axis="y", visible=False)
handles, names = axes[0].get_legend_handles_labels()
figure.legend(handles, names, frameon=False, ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.08))
figure.tight_layout()
show_figure(figure, "width-by-role")

## 5. Why our oracle ties Belletti-Simple

When only edges are checked, Logique's XAG oracle and the Belletti-Simple baseline are the same circuit. Both compare
each edge's colors with c − 1 AND gates and combine the |E| results with |E| − 1 more. Once Belletti-Simple's
multi-controlled gates are expanded with the manuscript's throwaway tree, the two have the same Toffoli gates, the
same T-count, 8(|E|c − 2) + 7, and the same width, (|V| + |E|)c − 1 qubits. The drawings show K₃: the registers `v`
hold the colors, `w` and `e` hold intermediate results, `a` the tree's ancilla and `out` the output.

In [ ]:
xag_k3 = logique_instance(replace(problems["K3"]), "xag", "R3", executable=EXECUTABLE)
simple_k3 = baseline_instance(problems["K3"], "simple", "R3")
for name, instance in (("XAG oracle", xag_k3), ("Belletti-Simple, throwaway tree", simple_k3)):
    circuit = expanded(instance)
    display(Markdown(f"**{name}**: {circuit.num_qubits} qubits, {costs.ft_counts(circuit)['t_count']} T gates"))
    show_figure(labelled(circuit, role_groups(instance, circuit)).draw("mpl", fold=45), f"k3-{instance.method}")

In [ ]:
equivalence = []
for g in GRAPHS:
    p = problems[g]
    c = max(1, (p.colors - 1).bit_length())
    pick = lambda m: ft[(ft.component == "oracle") & (ft.benchmark == g) & (ft.regime == "R3") & (ft.method == m) & (ft.mcx_policy == "throwaway_tree")].iloc[0]
    x, s = pick("xag"), pick("sb_simple")
    balanced = expanded(logique_instance(p, "xag", "R3", executable=EXECUTABLE, balanced=True))
    equivalence.append(
        {
            "graph": g,
            "T, XAG": int(x.t_count),
            "T, Belletti-Simple": int(s.t_count),
            "T, formula": 8 * (len(p.edges) * c - 2) + 7,
            "qubits, XAG": int(x.qubits),
            "qubits, Belletti-Simple": int(s.qubits),
            "qubits, formula": (len(p.vertices) + len(p.edges)) * c - 1,
            "T-depth, XAG": int(x.t_depth),
            "T-depth, XAG balanced": int(costs.ft_metrics(costs.clifford_t(balanced))["t_depth"]),
            "T-depth, Belletti-Simple": int(s.t_depth),
        }
    )
equivalence = pd.DataFrame(equivalence).set_index("graph")
display(equivalence)
assert (equivalence["T, XAG"] == equivalence["T, Belletti-Simple"]).all()
assert (equivalence["qubits, XAG"] == equivalence["qubits, Belletti-Simple"]).all()

The counts are identical; only the order of the gates differs. The XAG joins the edge results in a long chain, which
makes it deeper; building the conjunction as a balanced tree removes most of that difference. The width claims of the
manuscript therefore came from the ancilla counting of Section 2, not from the synthesis. Worse for Logique,
Belletti-Simple keeps its multi-controlled gates until decomposition, so other constructions can reuse ancillas and
make it narrower, at a moderate cost in T gates; the synthesized circuit fixes one qubit per AND gate.

In [ ]:
oracle_r3 = ft[(ft.component == "oracle") & (ft.regime == "R3")]
xag_r3 = oracle_r3[oracle_r3.method == "xag"].groupby("benchmark")[["qubits", "t_count"]].min()
simple_r3 = oracle_r3[oracle_r3.method == "sb_simple"].set_index(["benchmark", "mcx_policy"])[["qubits", "t_count"]]
trade = simple_r3.div(xag_r3, level="benchmark").reset_index()
figure, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
order = list(reversed(GRAPHS))
for ax, column, title in zip(axes, ("qubits", "t_count"), ("qubits", "T gates")):
    for offset, (policy_name, color) in zip(np.linspace(0.24, -0.24, len(POLICIES)), zip(POLICIES, SERIES)):
        values = trade[trade.mcx_policy == policy_name].set_index("benchmark")[column].reindex(order)
        ax.scatter(values, np.arange(len(order)) + offset, s=30, color=color, label=policy_name, zorder=2)
    ax.axvline(1, color=SECONDARY, linewidth=1)
    ax.set_xscale("log")
    log_axis(ax.xaxis)
    ax.set_xlabel(f"Belletti-Simple {title} / XAG")
    ax.grid(axis="y", visible=False)
axes[0].set_yticks(range(len(order)), order)
handles, names = axes[0].get_legend_handles_labels()
figure.legend(handles, names, frameon=False, ncol=5, loc="lower center", bbox_to_anchor=(0.5, -0.1))
figure.tight_layout()
show_figure(figure, "simple-policies")
KEY["simple narrowest"] = float(trade.qubits.min())

## 6. Optimizing the synthesized oracle

Three optimizations act on the synthesized oracle before it is mapped to gates. **Multiplicative-complexity
reduction** rewrites the logic with fewer AND gates; it helps where the predicate is more than a set of edge
comparisons, namely the checks of unused codes. **Balanced conjunctions** keep the gates but shorten the circuit.
**Caterpillar's mapping strategies** decide which intermediate results are kept or recomputed.

In [ ]:
reduction = []
for g in GRAPHS:
    if g in ("K20", "myciel5"):
        continue  # the pass is slow on the largest predicates
    encoding = encode_coloring(problems[g])
    plain = synthesize(encoding, "xag", executable=EXECUTABLE).summary()["AND"]
    reduced = synthesize(encoding, "xag", executable=EXECUTABLE, mc_reduction=True).summary()["AND"]
    reduction.append({"graph": g, "AND gates": plain, "after the pass": reduced, "removed": plain - reduced})
display(Markdown("AND gates of the complete predicate (R1 and R2) before and after the reduction pass"))
display(pd.DataFrame(reduction).set_index("graph"))

strategies = comparisons["t_count"]
strategy_view = pd.concat(
    {
        metric: comparisons[metric][comparisons[metric].regime == "R3"].pivot(index="benchmark", columns="method", values="ratio")
        for metric in ("t_count", "qubits_total")
    },
    axis=1,
).reindex(GRAPHS)
methods = [m for m in ("xag", "xag_lowt", "xag_fast_lowt", "xag_lowdepth", "xag_eager", "xag_bennett", "aig_bennett") if m in LOGIQUE]
display(Markdown("Caterpillar's strategies against the strongest baseline per iteration (R3): T gates, then qubits"))
display(strategy_view.loc[:, (slice(None), methods)].round(2))

## 7. Cheaper preparation of the valid colors

For every number of colors in the benchmarks (3, 4, 5, 6, 10 and 20), the preparation of a vertex needs **one**
y-rotation, which splits the amplitude 1 : 2 (k = 3 and 6) or 1 : 4 (k = 5, 10 and 20), plus Hadamard and controlled
Hadamard gates. The preparation and its inverse run in every iteration, so this rotation is paid 2j + 1 times per
vertex, and each copy must be approximated with Clifford+T gates to a share of the error budget. Four alternatives were
implemented and verified:

- **Exact rotations by repeat-until-success.** Both rotations can be implemented *exactly* by a small circuit with
  ancillas and a measurement: on success the rotation is applied, on failure the qubit is left unchanged and the
  attempt is repeated. For 1 : 2, an exhaustive search found an attempt with only 2 T gates that succeeds with
  probability 3/4; for 1 : 4, Jones's attempt uses one AND gate (4 T) and succeeds with probability 5/8. Exact
  rotations take no share of the error budget.
- **Better approximations** of each rotation, priced with the mean T-counts reported by Kliuchnikov et al. (2023):
  mixed diagonal, fallback (repeat-until-success), and mixed fallback, against Ross and Selinger's unitary method.
- **Hamming-weight phasing** (Gidney): the same rotation on n vertices costs about log₂ n rotations plus an adder
  network, using qubits of the oracle's workspace, which are free while the preparation runs.
- **Amplitude-amplified preparation** (Babbush et al.): Hadamard gates plus one exact round of amplitude
  amplification. It needs three rotations per vertex and a comparator, so it is not expected to be cheaper; it is
  included to check.

In [ ]:
rotation_rows = []
for k in range(2, 21):
    circuit = prep.shukla_vedula(k)
    exact = prep.with_exact_rotations(circuit)
    rus = [op for op in exact.count_ops() if op.startswith("rus")]
    rotation_rows.append(
        {
            "colors k": k,
            "rotations": costs.ft_metrics(costs.clifford_t(circuit))["rotation_count"],
            "exact circuit": ", ".join(rus) or ("none needed" if k & (k - 1) == 0 else "no"),
            "expected T (exact)": round(sum(prep.EXACT_ROTATIONS[int(r[6])].expected_t for r in rus), 2) if rus else None,
            "T per approximated rotation (Ross-Selinger, eps=1e-6)": costs.rotation_t_count(1e-6) if k & (k - 1) else None,
        }
    )
display(pd.DataFrame(rotation_rows).set_index("colors k"))

branch_rows = []
for m, rotation in prep.EXACT_ROTATIONS.items():
    unitary = Operator(rotation.attempt).data
    ancillas = rotation.attempt.num_qubits - 1
    for outcome in range(1 << ancillas):
        block = unitary[np.ix_([(outcome << 1) | t for t in (0, 1)], [0, 1])]
        p = float(np.real(np.trace(block.conj().T @ block)) / 2)
        if p > 1e-12:
            label = format(outcome, f"0{ancillas}b")
            branch_rows.append({"split": f"1 : {m - 1}", "ancilla outcome": label, "probability": round(p, 4),
                                "effect": "rotation applied" if label == rotation.success else "target unchanged, retry"})
display(pd.DataFrame(branch_rows))
for m, rotation in prep.EXACT_ROTATIONS.items():
    names = ["target", *[f"ancilla {i}" for i in range(1, rotation.attempt.num_qubits)]]
    groups = [(n.replace(" ", ""), [i]) for i, n in enumerate(names)]
    display(Markdown(f"**One attempt of the 1 : {m - 1} rotation**: {rotation.t_per_attempt} T gates, success "
                     f"probability {rotation.probability}, expected {rotation.expected_t:.2f} T gates. {rotation.reference}."))
    show_figure(labelled(rotation.attempt, groups).draw("mpl"), f"rus-{m}")

In [ ]:
layer = prep.hamming_weight_phasing(5, 0.4)
display(Markdown(f"**Hamming-weight phasing of five vertices**: {layer.count_ops().get('rz', 0)} rotations instead of 5, "
                 f"{prep.hamming_weight_ancillas(5)} ancillas"))
show_figure(layer.draw("mpl", fold=40), "hamming-weight")
counts = pd.DataFrame({"vertices": range(1, 48)})
counts["rotations"] = counts.vertices.map(lambda n: n.bit_length())
counts["ancillas"] = counts.vertices.map(prep.hamming_weight_ancillas)
figure, ax = plt.subplots(figsize=(6, 2.8))
ax.plot(counts.vertices, counts.vertices, color=SERIES[1], linewidth=2, label="direct rotations")
ax.plot(counts.vertices, counts.rotations, color=SERIES[0], linewidth=2, label="Hamming-weight phasing")
ax.set_xlabel("Vertices")
ax.set_ylabel("Rotations per layer")
ax.legend(frameon=False)
show_figure(figure, "hamming-weight-count")

amplified = prep.amplitude_amplified(3)
display(Markdown("**Amplitude-amplified preparation of three colors** (data wires, the extra rotated wire, the comparator flag)"))
show_figure(amplified.draw("mpl", fold=40), "amplitude-amplified")

The decisive test is the complete search. For each graph, the preparation and the diffusion operator are rebuilt with
each method and priced; the oracle is unchanged, so its cost comes from the study. The first row, the current
preparation with Ross and Selinger's rotations, reproduces the study's totals exactly. Rotations share the error
budget of 10⁻³ for the whole search.

In [ ]:
MODELS = list(costs.ROTATION_SYNTHESIS)
search_rows = []
for g in GRAPHS:
    p = problems[g]
    for regime in ("R2", "R3"):
        instance = logique_instance(p, "xag", regime, executable=EXECUTABLE)
        sel = ft[(ft.benchmark == g) & (ft.regime == regime) & (ft.method == "xag") & (ft.mcx_policy == "clean_tree")]
        oracle_t = float(sel[sel.component == "oracle"].t_count.iloc[0])
        found = sel[sel.component == "search"]
        j = int(found.iterations.iloc[0]) if len(found) else None
        for method in prep.PREPARATIONS:
            preparation_c, diffusion_c, _, clean = prepared.prepared_components(p, instance, method)
            counts_a = prepared.component_counts(preparation_c, "clean_tree", clean)
            counts_d = prepared.component_counts(diffusion_c, "clean_tree", clean)
            for model in MODELS:
                if method == "exact_rus" and model != "ross_selinger":
                    continue  # no approximated rotation is left
                row = {"graph": g, "regime": regime, "preparation": method, "rotations": model if method != "exact_rus" else "exact",
                       "qubits": diffusion_c.num_qubits}
                one = prepared.search_total(counts_a, counts_d, oracle_t, j or 1, model=model)
                row["T per iteration"] = (
                    oracle_t
                    + counts_d["t_count"]
                    + counts_d["rus_expected_t"]
                    + counts_d["rotation_count"] * one["t_per_rotation"]
                )
                if j is not None:
                    total = prepared.search_total(counts_a, counts_d, oracle_t, j, model=model)
                    row["T, complete search"] = total["t_total"]
                    row["iterations"] = j
                    if method == "shukla_vedula" and model == "ross_selinger":
                        assert math.isclose(total["t_total"], float(found.t_count.iloc[0])), (g, regime)
                search_rows.append(row)
choices = pd.DataFrame(search_rows)
r1 = search[(search.method == "xag") & (search.regime == "R1") & (search.mcx_policy == "clean_tree")].set_index("benchmark").t_count
current = choices[(choices.preparation == "shukla_vedula") & (choices.rotations == "ross_selinger")].set_index(["graph", "regime"])["T, complete search"]
choices["vs current"] = choices["T, complete search"] / choices.set_index(["graph", "regime"]).index.map(current)
display(
    choices[choices.regime == "R3"]
    .pivot_table(index=["preparation", "rotations"], columns="graph", values="vs current")
    .reindex(columns=[g for g in GRAPHS if g in set(choices.dropna(subset=["vs current"]).graph)])
    .round(2)
)

In [ ]:
best_choice = choices.dropna(subset=["T, complete search"])
labels_order = [
    ("shukla_vedula", "ross_selinger", "current preparation"),
    ("shukla_vedula", "mixed_fallback", "better approximations (mixed fallback)"),
    ("hamming_weight", "ross_selinger", "Hamming-weight phasing"),
    ("exact_rus", "exact", "exact rotations"),
    ("amplitude_amplified", "ross_selinger", "amplitude-amplified"),
]
counted = [g for g in GRAPHS if g in set(best_choice.graph)]
figure, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, regime in zip(axes, ("R2", "R3")):
    part = best_choice[best_choice.regime == regime]
    for i, ((method, model, label), color) in enumerate(zip(labels_order, [SERIES[1], SERIES[3], SERIES[2], SERIES[0], SERIES[4]])):
        values = part[(part.preparation == method) & (part.rotations == model)].set_index("graph")["T, complete search"].reindex(counted)
        ax.scatter(np.arange(len(counted)) + (i - 2) * 0.14, values / r1.reindex(counted).to_numpy(), s=34, color=color,
                   edgecolor="white", linewidth=0.8, label=label, zorder=2)
    ax.axhline(1, color=SECONDARY, linewidth=1, zorder=1, label="Hadamard start (R1)")
    ax.set_yscale("log")
    log_axis(ax.yaxis)
    ax.set_xticks(range(len(counted)), counted, rotation=45, ha="right")
    ax.set_title(f"{regime}: complete search / Hadamard start", color=INK)
    ax.grid(axis="x", visible=False)
handles, names = axes[0].get_legend_handles_labels()
figure.legend(handles, names, frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.14))
figure.tight_layout()
show_figure(figure, "preparations")

rotated = [g for g in counted if problems[g].colors & (problems[g].colors - 1)]
exact = best_choice[(best_choice.preparation == "exact_rus")].set_index(["graph", "regime"])["vs current"]
KEY["exact saving"] = {g: round(1 - exact[(g, "R3")], 2) for g in rotated}
winners = best_choice.loc[best_choice.groupby(["graph", "regime"])["T, complete search"].idxmin()]
KEY["best preparation"] = winners.groupby("preparation").size().to_dict()
r2 = best_choice[best_choice.regime == "R2"].set_index(["preparation", "rotations", "graph"])["T, complete search"]
KEY["flipped"] = [
    g for g in rotated
    if r2[("shukla_vedula", "ross_selinger", g)] >= r1[g] > r2[("exact_rus", "exact", g)]
]
KEY["valid start cheaper"] = [g for g in rotated if r2[("exact_rus", "exact", g)] < r1[g]]
display(Markdown("Cheapest preparation per graph and setting"))
display(winners.set_index(["graph", "regime"])[["preparation", "rotations", "T, complete search", "vs current"]].round(3))

Graphs whose number of colors is a power of two (myciel3, 2-Insertions_3 and zed-city_2019 have four colors) need no
rotation, so every method costs the same there. On the others, the exact rotations are the cheapest preparation: they
remove the approximated rotations altogether, at an expected cost of a few T gates per vertex. Hamming-weight phasing
comes next, and better approximations alone help less. Amplitude amplification is more expensive than the current
preparation, as expected from its three rotations per vertex. Starting in valid colors was already cheaper than the
Hadamard start except on K₃; with the exact rotations it is cheaper on every graph that needs rotations. What remains
of the diffusion operator's cost is its multi-controlled gate over all data qubits.

## 8. Key results and what remains

In [ ]:
ratios = KEY["xag ratios"]
lines = [
    f"- **Corrected comparison.** Per iteration, the XAG oracle needs {ratios[('t_count', 'R1')]:.2f}×, "
    f"{ratios[('t_count', 'R2')]:.2f}× and {ratios[('t_count', 'R3')]:.2f}× the T gates of the strongest baseline "
    f"in R1, R2 and R3 (geometric means), and between "
    f"{min(ratios[('qubits_total', r)] for r in REGIMES):.1f}× and "
    f"{max(ratios[('qubits_total', r)] for r in REGIMES):.1f}× its qubits.",
    "- **Same circuit as Belletti-Simple** when only edges are checked: equal T-count and width on all nine graphs. "
    f"Restoring decompositions make Belletti-Simple as narrow as {KEY['simple narrowest']:.2f}× the XAG oracle.",
    f"- **Measurement-based uncomputation** saves up to {max(KEY['measurement saving'].values()):.0%} of an oracle's T gates.",
    "- **Exact rotations** cut the complete edge-only search by "
    + ", ".join(f"{g} {v:.0%}" for g, v in KEY["exact saving"].items())
    + " (graphs with four colors need no rotation). Starting in valid colors becomes cheaper than the Hadamard start "
    + f"on {', '.join(KEY['valid start cheaper'])}; it was not before on {', '.join(KEY['flipped']) or 'none'}.",
    f"- **Diffusion fix.** On K₃, one iteration succeeds with probability "
    f"{KEY['k3 success']['reflection about the preparation']:.2f} instead of {KEY['k3 success']['historical Hadamard reflection']:.2f}.",
]
display(Markdown("\n".join(lines)))

**What remains.** Mapping the synthesized oracle under a qubit budget (SAT-based pebbling), to test whether synthesis
can reach the baseline's width automatically; a balancing tool and a study of all optimization profiles on the nine
graphs; success probability against the number of iterations, with noise; the scalability of synthesis on larger and
denser graphs, with the cost of the complete search set beside the growth of classical algorithms; and the writing of
the revised paper.

In [ ]:
from logique.benchmarks.provenance import checkout_revision, environment

overview.to_csv(OUT / "graphs.csv")
widths.to_csv(OUT / "width_accounting.csv")
summary.to_csv(OUT / "comparison_summary.csv", index=False)
equivalence.to_csv(OUT / "equivalence.csv")
choices.to_csv(OUT / "preparations.csv", index=False)
record = {
    "study": manifest["run_id"],
    "key_results": {k: (v if not isinstance(v, dict) else {str(a): b for a, b in v.items()}) for k, v in KEY.items()},
    "versions": environment(),
    "checkout": checkout_revision(ROOT),
}
(OUT / "overview_manifest.json").write_text(json.dumps(record, indent=2, default=str) + "\n")
print("Saved:", sorted(p.name for p in OUT.iterdir()))